# Random Forest CPCV

Tree-based metamodel workflow for Random Forest. It follows the logistic-regression notebook structure and evaluates the requested PCA and correlation-clustering feature treatments.

In [1]:
OUTPUT_SUBDIR = "random_forest_cpcv"

In [2]:
from __future__ import annotations

import itertools
import os
import sys
import tempfile
import warnings
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "matplotlib"))

import numpy as np
import pandas as pd
import matplotlib

if not os.environ.get("JPY_PARENT_PID"):
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 120)

RUN_FULL_GRID = False
SAVE_OUTPUTS = False
RANDOM_STATE = 42
PROBA_THRESHOLD = 0.50
TRAIN_END_DATE = pd.Timestamp("2022-01-01")
MIN_VALID_AUC_PATHS = 3

SMOKE_INSTRUMENTS = ["cl1s"]
FULL_INSTRUMENTS = ["cl1s", "ho1s", "rb1s", "ng1s"]
SMOKE_CONFIG_NAMES = ["ewma_10d_tp2_sl2"]

INSTRUMENTS = FULL_INSTRUMENTS if RUN_FULL_GRID else SMOKE_INSTRUMENTS

CPCV_SETTINGS = {
    "cl1s": {"n_groups": 6, "k_test_groups": 2},
    "rb1s": {"n_groups": 6, "k_test_groups": 2},
    "ho1s": {"n_groups": 4, "k_test_groups": 1},
    "ng1s": {"n_groups": 4, "k_test_groups": 1},
}

LEAKAGE_COLUMNS = {
    "training_end", "vol", "tp", "sl", "timeout_date", "timeout_close",
    "touch_date", "touch_price", "touched_barrier", "triple_barrier_label", "metalabel",
    "volatility_method", "ewma_span", "volatility_window", "num_days", "take_profit_mult",
    "stop_loss_mult", "holding_period_days", "raw_touch_return", "signed_touch_return",
}

ALWAYS_KEEP_FEATURES = ["primary_signal"]

FEATURE_SELECTION_CONFIGS = [
    {
        "name": "cluster_corr90",
        "method": "cluster",
        "corr_threshold": 0.90,
        "corr_method": "spearman",
        "selection_method": "target_corr",
        "max_features": 50,
    },
    {
        "name": "pca90",
        "method": "pca",
        "variance": 0.90,
        "standardize": True,
    },
]


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "features").exists():
            return candidate
    raise FileNotFoundError("Could not find project root with data/features")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
FEATURE_SELECTION_DIR = PROJECT_ROOT / "03_model_development" / "Feature Selction"
sys.path.insert(0, str(FEATURE_SELECTION_DIR))

from feature_selection_methods import CorrelationClusterSelector, PCAFeatureReducer

CLEAN_DIR = PROJECT_ROOT / "data" / "features" / "clean_feature_set"
TB_DIR = PROJECT_ROOT / "data" / "features" / "triple_barrier"
TB_SUMMARY_PATH = TB_DIR / "triple_barrier_config_summary.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "models" / OUTPUT_SUBDIR

print("Run full grid:", RUN_FULL_GRID)
print("Instruments:", INSTRUMENTS)
print("Feature selection configs:", [cfg["name"] for cfg in FEATURE_SELECTION_CONFIGS])
print("Feature selection module:", FEATURE_SELECTION_DIR)


Run full grid: False
Instruments: ['cl1s']
Feature selection configs: ['cluster_corr90', 'pca90']
Feature selection module: /Users/akshatg/Desktop/Imperial/Assessments/Term 3/systematic-strategies-with-machine-learning-Cw/03_model_development/Feature Selction


## 1. Define model grid

In [3]:
from sklearn.ensemble import RandomForestClassifier

if RUN_FULL_GRID:
    model_specs = [
        {
            "model_name": f"rf_t{n_estimators}_depth{max_depth}_leaf{min_samples_leaf}_mf{max_features}",
            "model_family": "random_forest",
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "max_features": max_features,
            "params": {
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "min_samples_leaf": min_samples_leaf,
                "max_features": max_features,
                "class_weight": "balanced_subsample",
                "bootstrap": True,
                "n_jobs": -1,
                "random_state": RANDOM_STATE,
            },
        }
        for n_estimators in [150, 300]
        for max_depth in [3, 5, 8]
        for min_samples_leaf in [5, 10]
        for max_features in ["sqrt", 0.5]
    ]
else:
    model_specs = [
        {
            "model_name": "rf_t120_depth4_leaf5_sqrt",
            "model_family": "random_forest",
            "n_estimators": 120,
            "max_depth": 4,
            "min_samples_leaf": 5,
            "max_features": "sqrt",
            "params": {
                "n_estimators": 120,
                "max_depth": 4,
                "min_samples_leaf": 5,
                "max_features": "sqrt",
                "class_weight": "balanced_subsample",
                "bootstrap": True,
                "n_jobs": -1,
                "random_state": RANDOM_STATE,
            },
        },
        {
            "model_name": "rf_t180_depth6_leaf10_mf50",
            "model_family": "random_forest",
            "n_estimators": 180,
            "max_depth": 6,
            "min_samples_leaf": 10,
            "max_features": 0.5,
            "params": {
                "n_estimators": 180,
                "max_depth": 6,
                "min_samples_leaf": 10,
                "max_features": 0.5,
                "class_weight": "balanced_subsample",
                "bootstrap": True,
                "n_jobs": -1,
                "random_state": RANDOM_STATE,
            },
        },
    ]

MODEL_PARAM_COLUMNS = ["n_estimators", "max_depth", "min_samples_leaf", "max_features"]


def make_estimator(spec: dict, y_train: pd.Series) -> RandomForestClassifier:
    del y_train
    return RandomForestClassifier(**spec["params"])


model_grid = pd.DataFrame([{k: v for k, v in spec.items() if k != "params"} for spec in model_specs])
display(model_grid)


,model_name,model_family,n_estimators,max_depth,min_samples_leaf,max_features
0,rf_t120_depth4_leaf5_sqrt,random_forest,120,4,5,sqrt
1,rf_t180_depth6_leaf10_mf50,random_forest,180,6,10,0.5


## 2. Load clean features and label files

Smoke mode loads one energy instrument and one triple-barrier configuration. Set `RUN_FULL_GRID = True` to run the full energy grid.

In [4]:
clean_features = {}

for instrument in INSTRUMENTS:
    path = CLEAN_DIR / f"{instrument}_clean_feature_set.csv"
    assert path.exists(), f"Missing clean feature file: {path}"

    df = pd.read_csv(path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    assert df["instrument"].eq(instrument).all()
    assert df["date"].duplicated().sum() == 0

    model_feature_cols = [c for c in df.columns if c not in {"date", "instrument", "primary_signal"}]
    assert df[model_feature_cols].isna().sum().sum() == 0, f"Missing feature values remain in {path}"
    clean_features[instrument] = df.drop(columns=["primary_signal"], errors="ignore")

label_files = pd.read_csv(TB_SUMMARY_PATH)
label_files = label_files[label_files["instrument"].isin(INSTRUMENTS)].copy()

if not RUN_FULL_GRID:
    label_files = label_files[label_files["config_name"].isin(SMOKE_CONFIG_NAMES)].copy()

label_files["label_path"] = label_files["output_path"].map(lambda p: str(Path(p) if Path(p).exists() else TB_DIR / Path(p).name))
missing = [p for p in label_files["label_path"] if not Path(p).exists()]
assert not missing, f"Missing label files: {missing[:5]}"
assert not label_files.empty, "No label files selected."

metalabel_distribution = label_files.assign(
    metalabel_0_count=label_files["metalabel_0_count"].astype(int),
    metalabel_1_count=label_files["metalabel_1_count"].astype(int),
    metalabel_0_rate=label_files["metalabel_0_count"] / label_files["rows"],
    metalabel_1_rate=label_files["metalabel_1_count"] / label_files["rows"],
)

display(pd.DataFrame({"instrument": k, "clean_rows": len(v), "clean_columns": v.shape[1]} for k, v in clean_features.items()))
display(
    metalabel_distribution[
        [
            "instrument", "config_name", "rows", "metalabel_0_count", "metalabel_1_count",
            "metalabel_0_rate", "metalabel_1_rate", "num_days", "label_path",
        ]
    ].head(20)
)


,instrument,clean_rows,clean_columns
0,cl1s,626,145


,instrument,config_name,rows,metalabel_0_count,metalabel_1_count,metalabel_0_rate,metalabel_1_rate,num_days,label_path
0,cl1s,ewma_10d_tp2_sl2,324,201,123,0.62037,0.37963,10,/Users/akshatg/Desktop/Imperial/Assessments/Te...


## 3. Build modelling data and CPCV splits

The split logic mirrors the logistic-regression notebook: purge overlapping event windows and embargo observations after each test block using the label file's `num_days`.

In [5]:
def read_labels(label_path: str) -> pd.DataFrame:
    columns = pd.read_csv(label_path, nrows=0).columns
    date_columns = [c for c in ["date", "training_end", "timeout_date", "touch_date"] if c in columns]
    return pd.read_csv(label_path, parse_dates=date_columns).sort_values("date").reset_index(drop=True)


def make_model_data(label_row: pd.Series):
    labels = read_labels(label_row["label_path"])
    features = clean_features[label_row["instrument"]]

    labels = labels.drop(columns=["close"], errors="ignore")
    data = labels.merge(features, on=["date", "instrument"], how="inner")
    data = data.sort_values("date").reset_index(drop=True)
    data = data[data["date"] < TRAIN_END_DATE].copy()
    assert "primary_signal" in data.columns, "primary_signal missing after merge."
    data = data[(data["metalabel"].isin([0, 1])) & (data["primary_signal"] != 0)].copy()
    data = data.sort_values("date").reset_index(drop=True)

    assert not data.empty, f"No merged rows for {label_row['instrument']} {label_row['config_name']}"
    assert data["metalabel"].nunique() == 2, "Need both metalabel classes."
    assert data["num_days"].nunique() == 1, "One label file should have one num_days value."

    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    feature_cols = [c for c in numeric_cols if c not in LEAKAGE_COLUMNS]
    assert not set(feature_cols).intersection(LEAKAGE_COLUMNS)
    assert feature_cols, "No numeric features available."
    assert "primary_signal" in feature_cols, "primary_signal should be available as a core metamodel feature."

    data[feature_cols] = data[feature_cols].replace([np.inf, -np.inf], np.nan)
    embargo_observations = int(data["num_days"].iloc[0])
    return data, feature_cols, embargo_observations


def date_after_n_observations(dates: pd.Series, end_date: pd.Timestamp, n: int) -> pd.Timestamp:
    unique_dates = pd.Series(pd.to_datetime(dates).sort_values().unique())
    first_after_end = unique_dates.searchsorted(pd.Timestamp(end_date), side="right")
    if first_after_end >= len(unique_dates):
        return pd.Timestamp(end_date)
    return pd.Timestamp(unique_dates.iloc[min(len(unique_dates) - 1, first_after_end + n - 1)])


def make_cpcv_splits(data: pd.DataFrame, instrument: str, embargo_observations: int) -> list[dict]:
    settings = CPCV_SETTINGS[instrument]
    groups = [g.astype(int) for g in np.array_split(np.arange(len(data)), settings["n_groups"]) if len(g) > 0]
    splits = []

    for split_id, test_group_ids in enumerate(itertools.combinations(range(len(groups)), settings["k_test_groups"])):
        test_idx = np.concatenate([groups[group_id] for group_id in test_group_ids])
        train_mask = np.ones(len(data), dtype=bool)
        train_mask[test_idx] = False

        for group_id in test_group_ids:
            group_idx = groups[group_id]
            test_start = data.loc[group_idx, "date"].min()
            test_end = data.loc[group_idx, "date"].max()
            test_horizon_end = data.loc[group_idx, "timeout_date"].max()
            embargo_end = date_after_n_observations(data["date"], test_end, embargo_observations)

            overlaps_test_window = (data["date"] <= test_horizon_end) & (data["timeout_date"] >= test_start)
            inside_embargo = (data["date"] > test_end) & (data["date"] <= embargo_end)
            train_mask[overlaps_test_window | inside_embargo] = False

        train_idx = np.where(train_mask)[0]
        assert set(train_idx).isdisjoint(set(test_idx)), "Train/test overlap found."

        if len(train_idx) > 0:
            splits.append({
                "split_id": split_id,
                "instrument": instrument,
                "test_groups": test_group_ids,
                "train_idx": train_idx,
                "test_idx": test_idx,
                "train_rows": len(train_idx),
                "test_rows": len(test_idx),
                "embargo_observations": embargo_observations,
            })

    return splits


def split_status_frame(data: pd.DataFrame, split: dict) -> pd.DataFrame:
    status = pd.Series("train", index=data.index)
    status.iloc[split["test_idx"]] = "test"

    for group_id in split["test_groups"]:
        settings = CPCV_SETTINGS[split["instrument"]]
        groups = [g.astype(int) for g in np.array_split(np.arange(len(data)), settings["n_groups"]) if len(g) > 0]
        group_idx = groups[group_id]

        test_start = data.loc[group_idx, "date"].min()
        test_end = data.loc[group_idx, "date"].max()
        test_horizon_end = data.loc[group_idx, "timeout_date"].max()
        embargo_end = date_after_n_observations(data["date"], test_end, split["embargo_observations"])

        purged = (data["date"] <= test_horizon_end) & (data["timeout_date"] >= test_start)
        embargoed = (data["date"] > test_end) & (data["date"] <= embargo_end)

        status.loc[purged & status.ne("test")] = "purged"
        status.loc[embargoed & status.ne("test")] = "embargo"

    status.iloc[split["train_idx"]] = "train"
    return pd.DataFrame({"date": data["date"], "split_id": split["split_id"], "status": status.values})


def plot_cpcv_splits(data: pd.DataFrame, splits: list[dict]) -> None:
    plot_df = pd.concat([split_status_frame(data, split) for split in splits], ignore_index=True)
    colours = {"train": "lightgrey", "purged": "#d62728", "embargo": "#ff7f0e", "test": "#1f77b4"}

    plt.figure(figsize=(14, 6))
    for status_name, status_df in plot_df.groupby("status"):
        plt.scatter(
            status_df["date"],
            status_df["split_id"],
            s=18,
            marker="s",
            color=colours[status_name],
            label=status_name,
            alpha=0.9,
        )

    plt.title("CPCV splits with purging and num_days embargo")
    plt.xlabel("Event date")
    plt.ylabel("Split id")
    plt.yticks(sorted(plot_df["split_id"].unique()))
    plt.legend(loc="upper right")
    plt.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    if os.environ.get("JPY_PARENT_PID"):
        plt.show()
    else:
        plt.close()


example_data, example_features, example_embargo = make_model_data(label_files.iloc[0])
example_splits = make_cpcv_splits(example_data, label_files.iloc[0]["instrument"], example_embargo)
print("Example rows:", len(example_data))
print("Example raw features:", len(example_features))
print("Example embargo observations:", example_embargo)
print("Example CPCV splits:", len(example_splits))
plot_cpcv_splits(example_data, example_splits)


Example rows: 323
Example raw features: 144
Example embargo observations: 10
Example CPCV splits: 15


## 4. Feature selection and reduction

Both requested methods are fitted inside each CPCV training fold only, then applied to the matching test fold: correlation clustering and PCA.

In [6]:
def target_correlation_ranking(X: pd.DataFrame, y: pd.Series, method: str = "spearman") -> pd.Series:
    scores = {}
    y = pd.Series(y, index=X.index, name="target")
    for col in X.columns:
        pair = pd.concat([pd.to_numeric(X[col], errors="coerce"), y], axis=1).dropna()
        if pair.shape[0] < 3 or pair.iloc[:, 0].nunique() <= 1 or pair.iloc[:, 1].nunique() <= 1:
            scores[col] = np.nan
        else:
            scores[col] = abs(pair.iloc[:, 0].corr(pair.iloc[:, 1], method=method))
    return pd.Series(scores, name="target_abs_corr").sort_values(ascending=False, na_position="last")


def always_keep_feature_names(feature_names: list[str]) -> list[str]:
    return [feature for feature in ALWAYS_KEEP_FEATURES if feature in feature_names]


def ranked_feature_cap(feature_names: list[str], ranking: pd.Series, max_features: int | None) -> list[str]:
    if max_features is None or len(feature_names) <= max_features:
        return feature_names

    always_keep = always_keep_feature_names(feature_names)
    remaining_slots = max(max_features - len(always_keep), 0)
    ranked_candidates = [feature for feature in ranking.index if feature in feature_names and feature not in always_keep]
    selected = always_keep + ranked_candidates[:remaining_slots]
    return list(dict.fromkeys(selected))


def append_raw_keep_features(
    X_train: np.ndarray,
    X_test: np.ndarray,
    train: pd.DataFrame,
    test: pd.DataFrame,
    transformed_feature_names: list[str],
) -> tuple[np.ndarray, np.ndarray, list[str], SimpleImputer | None]:
    raw_keep = [feature for feature in ALWAYS_KEEP_FEATURES if feature in train.columns and feature not in transformed_feature_names]
    if not raw_keep:
        return X_train, X_test, transformed_feature_names, None

    imputer = SimpleImputer(strategy="median")
    train_keep = imputer.fit_transform(train[raw_keep].replace([np.inf, -np.inf], np.nan))
    test_keep = imputer.transform(test[raw_keep].replace([np.inf, -np.inf], np.nan))
    return (
        np.column_stack([X_train, train_keep]),
        np.column_stack([X_test, test_keep]),
        transformed_feature_names + raw_keep,
        imputer,
    )


def prepare_fold_features(
    train: pd.DataFrame,
    test: pd.DataFrame,
    feature_cols: list[str],
    y_train: pd.Series,
    feature_config: dict,
):
    method = feature_config["method"]
    X_train_base = train[feature_cols].replace([np.inf, -np.inf], np.nan)
    X_test_base = test[feature_cols].replace([np.inf, -np.inf], np.nan)

    if method == "cluster":
        selector = CorrelationClusterSelector(
            corr_threshold=feature_config.get("corr_threshold", 0.90),
            corr_method=feature_config.get("corr_method", "spearman"),
            selection_method=feature_config.get("selection_method", "target_corr"),
            min_periods=20,
        )
        selector.fit(X_train_base, y=y_train, feature_columns=feature_cols)
        selected_features = selector.selected_features_

        max_features = feature_config.get("max_features")
        if max_features is not None and len(selected_features) > max_features:
            ranking = target_correlation_ranking(X_train_base[selected_features], y_train, method=feature_config.get("corr_method", "spearman"))
            selected_features = ranked_feature_cap(selected_features, ranking, max_features)

        imputer = SimpleImputer(strategy="median")
        X_train = imputer.fit_transform(X_train_base[selected_features])
        X_test = imputer.transform(X_test_base[selected_features])

        metadata = {
            "method": method,
            "selector": selector,
            "imputer": imputer,
            "raw_feature_count": len(feature_cols),
            "transformed_feature_count": len(selected_features),
            "feature_names": selected_features,
            "cluster_count": len(selector.cluster_summary_),
            "dropped_feature_count": len(selector.dropped_features_),
        }
        return X_train, X_test, selected_features, metadata

    if method == "pca":
        reducer = PCAFeatureReducer(
            n_components=feature_config.get("variance", 0.90),
            standardize=feature_config.get("standardize", True),
            component_prefix="pca",
            random_state=RANDOM_STATE,
        )
        reducer.fit(X_train_base, feature_columns=feature_cols)
        X_train_df = reducer.transform(X_train_base)
        X_test_df = reducer.transform(X_test_base)
        X_train, X_test, transformed_feature_names, raw_keep_imputer = append_raw_keep_features(
            X_train_df.to_numpy(),
            X_test_df.to_numpy(),
            train=X_train_base,
            test=X_test_base,
            transformed_feature_names=reducer.component_columns_,
        )

        metadata = {
            "method": method,
            "reducer": reducer,
            "raw_keep_imputer": raw_keep_imputer,
            "raw_keep_features": [feature for feature in transformed_feature_names if feature in ALWAYS_KEEP_FEATURES],
            "raw_feature_count": len(feature_cols),
            "transformed_feature_count": len(transformed_feature_names),
            "feature_names": transformed_feature_names,
            "explained_variance": float(reducer.explained_variance_ratio_.sum()),
        }
        return X_train, X_test, transformed_feature_names, metadata

    raise ValueError(f"Unknown feature-selection method: {method}")


def get_preprocessor_summary(metadata: dict) -> dict:
    summary = {
        "raw_feature_count": metadata["raw_feature_count"],
        "feature_count": metadata["transformed_feature_count"],
    }
    if metadata["method"] == "cluster":
        summary.update(
            cluster_count=metadata["cluster_count"],
            dropped_feature_count=metadata["dropped_feature_count"],
            explained_variance=np.nan,
        )
    elif metadata["method"] == "pca":
        summary.update(
            cluster_count=np.nan,
            dropped_feature_count=np.nan,
            explained_variance=metadata["explained_variance"],
        )
    return summary


## 5. Run CPCV

Each model specification is crossed with the two feature configurations. The notebook stores path-level metrics and out-of-fold probabilities for threshold analysis.

In [7]:
def sharpe_ratio(returns: pd.Series) -> float:
    returns = pd.Series(returns).dropna()
    if len(returns) < 2 or returns.std(ddof=1) == 0:
        return np.nan
    return float(returns.mean() / returns.std(ddof=1) * np.sqrt(len(returns)))


def safe_auc(y_true: pd.Series, y_score: np.ndarray) -> float:
    return roc_auc_score(y_true, y_score) if pd.Series(y_true).nunique() == 2 else np.nan


results = []
baseline_results = []
prediction_rows = []
total_candidates = len(label_files) * len(FEATURE_SELECTION_CONFIGS) * len(model_specs)
candidate_number = 0

for _, label_row in label_files.iterrows():
    data, feature_cols, embargo_observations = make_model_data(label_row)
    splits = make_cpcv_splits(data, label_row["instrument"], embargo_observations)

    for split in splits:
        test = data.iloc[split["test_idx"]]
        y_test = test["metalabel"].astype(int)
        blind_score = np.ones(len(y_test), dtype=float)
        blind_pred = np.ones(len(y_test), dtype=int)
        baseline_results.append({
            "instrument": label_row["instrument"],
            "config_name": label_row["config_name"],
            "split_id": split["split_id"],
            "train_rows": split["train_rows"],
            "test_rows": split["test_rows"],
            "auc": safe_auc(y_test, blind_score),
            "brier": brier_score_loss(y_test, blind_score),
            "log_loss": log_loss(y_test, blind_score, labels=[0, 1]),
            "accuracy": accuracy_score(y_test, blind_pred),
            "precision": precision_score(y_test, blind_pred, zero_division=0),
            "recall": recall_score(y_test, blind_pred, zero_division=0),
            "f1": f1_score(y_test, blind_pred, zero_division=0),
            "trade_count": int(blind_pred.sum()),
            "sharpe": sharpe_ratio(test["signed_touch_return"]),
            "label_rows": len(data),
            "embargo_observations": embargo_observations,
        })

    for feature_config in FEATURE_SELECTION_CONFIGS:
        for spec in model_specs:
            candidate_number += 1
            print(
                f"{candidate_number}/{total_candidates}: "
                f"{label_row['instrument']} | {label_row['config_name']} | "
                f"{feature_config['name']} | {spec['model_name']}"
            )

            for split in splits:
                train = data.iloc[split["train_idx"]]
                test = data.iloc[split["test_idx"]]
                y_train = train["metalabel"].astype(int)
                y_test = test["metalabel"].astype(int)

                if y_train.nunique() < 2:
                    continue

                X_train, X_test, transformed_feature_names, prep_metadata = prepare_fold_features(
                    train=train,
                    test=test,
                    feature_cols=feature_cols,
                    y_train=y_train,
                    feature_config=feature_config,
                )

                model = make_estimator(spec, y_train)
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always", RuntimeWarning)
                    model.fit(X_train, y_train)
                    y_score = model.predict_proba(X_test)[:, 1]

                runtime_warnings = sum(issubclass(w.category, RuntimeWarning) for w in caught)
                y_pred = (y_score >= PROBA_THRESHOLD).astype(int)
                predicted_trade_returns = test.loc[y_pred == 1, "signed_touch_return"]
                prep_summary = get_preprocessor_summary(prep_metadata)

                result_row = {
                    "instrument": label_row["instrument"],
                    "config_name": label_row["config_name"],
                    "feature_config": feature_config["name"],
                    "feature_method": feature_config["method"],
                    "model_name": spec["model_name"],
                    "model_family": spec["model_family"],
                    "split_id": split["split_id"],
                    "train_rows": split["train_rows"],
                    "test_rows": split["test_rows"],
                    "embargo_observations": embargo_observations,
                    "auc": safe_auc(y_test, y_score),
                    "brier": brier_score_loss(y_test, y_score),
                    "log_loss": log_loss(y_test, y_score, labels=[0, 1]),
                    "accuracy": accuracy_score(y_test, y_pred),
                    "precision": precision_score(y_test, y_pred, zero_division=0),
                    "recall": recall_score(y_test, y_pred, zero_division=0),
                    "f1": f1_score(y_test, y_pred, zero_division=0),
                    "trade_count": int((y_pred == 1).sum()),
                    "sharpe": sharpe_ratio(predicted_trade_returns),
                    "runtime_warning_count": int(runtime_warnings),
                    "label_rows": len(data),
                    **{k: spec.get(k, np.nan) for k in MODEL_PARAM_COLUMNS},
                    **prep_summary,
                }
                results.append(result_row)

                prediction_rows.extend(
                    {
                        "instrument": label_row["instrument"],
                        "config_name": label_row["config_name"],
                        "feature_config": feature_config["name"],
                        "feature_method": feature_config["method"],
                        "model_name": spec["model_name"],
                        "model_family": spec["model_family"],
                        "split_id": split["split_id"],
                        "date": date,
                        "y_true": int(y_true),
                        "y_score": float(score),
                        "signed_touch_return": float(ret) if pd.notna(ret) else np.nan,
                    }
                    for date, y_true, score, ret in zip(test["date"], y_test, y_score, test["signed_touch_return"])
                )

path_level_results = pd.DataFrame(results)
baseline_path_results = pd.DataFrame(baseline_results)
oof_predictions = pd.DataFrame(prediction_rows)
assert not path_level_results.empty
assert not oof_predictions.empty

display(path_level_results.head())
display(baseline_path_results.head())
display(oof_predictions.head())


1/4: cl1s | ewma_10d_tp2_sl2 | cluster_corr90 | rf_t120_depth4_leaf5_sqrt
2/4: cl1s | ewma_10d_tp2_sl2 | cluster_corr90 | rf_t180_depth6_leaf10_mf50
3/4: cl1s | ewma_10d_tp2_sl2 | pca90 | rf_t120_depth4_leaf5_sqrt
4/4: cl1s | ewma_10d_tp2_sl2 | pca90 | rf_t180_depth6_leaf10_mf50


,instrument,config_name,feature_config,feature_method,model_name,model_family,split_id,train_rows,test_rows,embargo_observations,auc,brier,log_loss,accuracy,precision,recall,f1,trade_count,sharpe,runtime_warning_count,label_rows,n_estimators,max_depth,min_samples_leaf,max_features,raw_feature_count,feature_count,cluster_count,dropped_feature_count,explained_variance
0,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0,205,108,10,0.809023,0.208949,0.606254,0.675926,0.523077,0.894737,0.660194,65,4.866734,0,323,120,4,5,sqrt,144,50,13.0,77.0,NaN
1,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,1,188,108,10,0.595803,0.248352,0.691856,0.564815,0.553191,0.912281,0.688742,94,5.060200,0,323,120,4,5,sqrt,144,50,13.0,76.0,NaN
2,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,2,188,108,10,0.669690,0.235483,0.668712,0.638889,0.703704,0.622951,0.660870,54,4.823653,0,323,120,4,5,sqrt,144,50,17.0,79.0,NaN
3,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,3,187,108,10,0.461806,0.296524,0.800571,0.472222,0.423729,0.520833,0.467290,59,-0.261537,0,323,120,4,5,sqrt,144,50,15.0,75.0,NaN
4,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,4,196,107,10,0.619718,0.258403,0.709597,0.485981,0.722222,0.366197,0.485981,36,2.540521,0,323,120,4,5,sqrt,144,50,13.0,76.0,NaN


,instrument,config_name,split_id,train_rows,test_rows,auc,brier,log_loss,accuracy,precision,recall,f1,trade_count,sharpe,label_rows,embargo_observations
0,cl1s,ewma_10d_tp2_sl2,0,205,108,0.5,0.648148,23.361627,0.351852,0.351852,1.0,0.520548,108,4.392671,323,10
1,cl1s,ewma_10d_tp2_sl2,1,188,108,0.5,0.472222,17.020614,0.527778,0.527778,1.0,0.690909,108,5.462672,323,10
2,cl1s,ewma_10d_tp2_sl2,2,188,108,0.5,0.435185,15.685664,0.564815,0.564815,1.0,0.721893,108,4.850500,323,10
3,cl1s,ewma_10d_tp2_sl2,3,187,108,0.5,0.555556,20.024252,0.444444,0.444444,1.0,0.615385,108,3.368973,323,10
4,cl1s,ewma_10d_tp2_sl2,4,196,107,0.5,0.336449,12.126837,0.663551,0.663551,1.0,0.797753,107,4.774000,323,10


,instrument,config_name,feature_config,feature_method,model_name,model_family,split_id,date,y_true,y_score,signed_touch_return
0,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0,2020-01-07,1,0.319703,0.049282
1,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0,2020-01-22,1,0.512380,0.044942
2,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0,2020-01-23,1,0.501010,0.044073
3,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0,2020-01-24,1,0.630431,0.048533
4,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0,2020-01-27,1,0.585893,0.057019


## 6. Summarise candidates and compare with blind primary-signal baseline

In [8]:
group_cols = [
    "instrument", "config_name", "feature_config", "feature_method",
    "model_name", "model_family", *MODEL_PARAM_COLUMNS,
]

candidate_summary = (
    path_level_results
    .groupby(group_cols, dropna=False, as_index=False)
    .agg(
        mean_auc=("auc", "mean"),
        median_auc=("auc", "median"),
        std_auc=("auc", "std"),
        valid_auc_paths=("auc", lambda x: x.notna().sum()),
        mean_brier=("brier", "mean"),
        mean_log_loss=("log_loss", "mean"),
        mean_accuracy=("accuracy", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_f1=("f1", "mean"),
        median_sharpe=("sharpe", "median"),
        mean_sharpe=("sharpe", "mean"),
        total_trades=("trade_count", "sum"),
        mean_feature_count=("feature_count", "mean"),
        mean_raw_feature_count=("raw_feature_count", "mean"),
        mean_cluster_count=("cluster_count", "mean"),
        mean_dropped_feature_count=("dropped_feature_count", "mean"),
        mean_explained_variance=("explained_variance", "mean"),
        total_runtime_warnings=("runtime_warning_count", "sum"),
        label_rows=("label_rows", "first"),
        embargo_observations=("embargo_observations", "first"),
    )
)

max_valid_paths = int(candidate_summary["valid_auc_paths"].max())
min_paths = min(MIN_VALID_AUC_PATHS, max_valid_paths)
candidate_summary = candidate_summary[candidate_summary["valid_auc_paths"] >= min_paths].copy()

candidate_summary = candidate_summary.sort_values(
    ["instrument", "mean_auc", "std_auc", "median_sharpe", "total_trades"],
    ascending=[True, False, True, False, False],
).reset_index(drop=True)

best_by_instrument = candidate_summary.groupby("instrument", as_index=False, group_keys=False).head(1).reset_index(drop=True)
best_by_instrument_config = candidate_summary.groupby(["instrument", "config_name"], as_index=False, group_keys=False).head(1).reset_index(drop=True)

baseline_summary = (
    baseline_path_results
    .groupby(["instrument", "config_name"], as_index=False)
    .agg(
        baseline_mean_auc=("auc", "mean"),
        baseline_mean_brier=("brier", "mean"),
        baseline_mean_log_loss=("log_loss", "mean"),
        baseline_mean_accuracy=("accuracy", "mean"),
        baseline_mean_precision=("precision", "mean"),
        baseline_mean_recall=("recall", "mean"),
        baseline_mean_f1=("f1", "mean"),
        baseline_median_sharpe=("sharpe", "median"),
        baseline_total_trades=("trade_count", "sum"),
    )
)

comparison_to_baseline = candidate_summary.merge(baseline_summary, on=["instrument", "config_name"], how="left")
comparison_to_baseline["auc_lift_vs_blind"] = comparison_to_baseline["mean_auc"] - comparison_to_baseline["baseline_mean_auc"]
comparison_to_baseline["f1_lift_vs_blind"] = comparison_to_baseline["mean_f1"] - comparison_to_baseline["baseline_mean_f1"]

display(candidate_summary.head(30))
display(best_by_instrument)
display(comparison_to_baseline.head(30))


,instrument,config_name,feature_config,feature_method,model_name,model_family,n_estimators,max_depth,min_samples_leaf,max_features,mean_auc,median_auc,std_auc,valid_auc_paths,mean_brier,mean_log_loss,mean_accuracy,mean_precision,mean_recall,mean_f1,median_sharpe,mean_sharpe,total_trades,mean_feature_count,mean_raw_feature_count,mean_cluster_count,mean_dropped_feature_count,mean_explained_variance,total_runtime_warnings,label_rows,embargo_observations
0,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t180_depth6_leaf10_mf50,random_forest,180,6,10,0.5,0.593392,0.606906,0.133525,15,0.272561,0.753214,0.545321,0.429665,0.508907,0.406552,3.368278,2.815669,727,50.0,144.0,14.2,79.0,NaN,0,323,10
1,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,120,4,5,sqrt,0.593359,0.619718,0.138505,15,0.260102,0.723885,0.561988,0.459018,0.516785,0.425394,3.615462,3.168885,706,50.0,144.0,14.2,79.0,NaN,0,323,10
2,cl1s,ewma_10d_tp2_sl2,pca90,pca,rf_t120_depth4_leaf5_sqrt,random_forest,120,4,5,sqrt,0.545292,0.534141,0.120240,15,0.261738,0.720234,0.518836,0.412258,0.455628,0.376403,1.579064,1.350662,664,14.8,144.0,NaN,NaN,0.90514,0,323,10
3,cl1s,ewma_10d_tp2_sl2,pca90,pca,rf_t180_depth6_leaf10_mf50,random_forest,180,6,10,0.5,0.543541,0.526208,0.113562,15,0.269975,0.740477,0.521934,0.420753,0.522044,0.409851,2.590095,2.045467,753,14.8,144.0,NaN,NaN,0.90514,0,323,10


,instrument,config_name,feature_config,feature_method,model_name,model_family,n_estimators,max_depth,min_samples_leaf,max_features,mean_auc,median_auc,std_auc,valid_auc_paths,mean_brier,mean_log_loss,mean_accuracy,mean_precision,mean_recall,mean_f1,median_sharpe,mean_sharpe,total_trades,mean_feature_count,mean_raw_feature_count,mean_cluster_count,mean_dropped_feature_count,mean_explained_variance,total_runtime_warnings,label_rows,embargo_observations
0,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t180_depth6_leaf10_mf50,random_forest,180,6,10,0.5,0.593392,0.606906,0.133525,15,0.272561,0.753214,0.545321,0.429665,0.508907,0.406552,3.368278,2.815669,727,50.0,144.0,14.2,79.0,NaN,0,323,10


,instrument,config_name,feature_config,feature_method,model_name,model_family,n_estimators,max_depth,min_samples_leaf,max_features,mean_auc,median_auc,std_auc,valid_auc_paths,mean_brier,mean_log_loss,mean_accuracy,mean_precision,mean_recall,mean_f1,median_sharpe,mean_sharpe,total_trades,mean_feature_count,mean_raw_feature_count,mean_cluster_count,mean_dropped_feature_count,mean_explained_variance,total_runtime_warnings,label_rows,embargo_observations,baseline_mean_auc,baseline_mean_brier,baseline_mean_log_loss,baseline_mean_accuracy,baseline_mean_precision,baseline_mean_recall,baseline_mean_f1,baseline_median_sharpe,baseline_total_trades,auc_lift_vs_blind,f1_lift_vs_blind
0,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t180_depth6_leaf10_mf50,random_forest,180,6,10,0.5,0.593392,0.606906,0.133525,15,0.272561,0.753214,0.545321,0.429665,0.508907,0.406552,3.368278,2.815669,727,50.0,144.0,14.2,79.0,NaN,0,323,10,0.5,0.618899,22.307391,0.381101,0.381101,1.0,0.533583,2.713413,1615,0.093392,-0.127031
1,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,120,4,5,sqrt,0.593359,0.619718,0.138505,15,0.260102,0.723885,0.561988,0.459018,0.516785,0.425394,3.615462,3.168885,706,50.0,144.0,14.2,79.0,NaN,0,323,10,0.5,0.618899,22.307391,0.381101,0.381101,1.0,0.533583,2.713413,1615,0.093359,-0.108189
2,cl1s,ewma_10d_tp2_sl2,pca90,pca,rf_t120_depth4_leaf5_sqrt,random_forest,120,4,5,sqrt,0.545292,0.534141,0.120240,15,0.261738,0.720234,0.518836,0.412258,0.455628,0.376403,1.579064,1.350662,664,14.8,144.0,NaN,NaN,0.90514,0,323,10,0.5,0.618899,22.307391,0.381101,0.381101,1.0,0.533583,2.713413,1615,0.045292,-0.157180
3,cl1s,ewma_10d_tp2_sl2,pca90,pca,rf_t180_depth6_leaf10_mf50,random_forest,180,6,10,0.5,0.543541,0.526208,0.113562,15,0.269975,0.740477,0.521934,0.420753,0.522044,0.409851,2.590095,2.045467,753,14.8,144.0,NaN,NaN,0.90514,0,323,10,0.5,0.618899,22.307391,0.381101,0.381101,1.0,0.533583,2.713413,1615,0.043541,-0.123732


## 7. Decision-threshold analysis

Use out-of-fold probabilities to compare precision, recall, F1, trade count, Sharpe, and confusion-matrix counts across thresholds.

In [9]:
def threshold_metric_rows(group: pd.DataFrame, thresholds: np.ndarray) -> list[dict]:
    rows = []
    y_true = group["y_true"].astype(int).to_numpy()
    y_score = group["y_score"].astype(float).to_numpy()
    returns = group["signed_touch_return"].to_numpy()

    for threshold in thresholds:
        y_pred = (y_score >= threshold).astype(int)
        trade_returns = returns[y_pred == 1]
        rows.append({
            "threshold": round(float(threshold), 2),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "trade_count": int(y_pred.sum()),
            "sharpe": sharpe_ratio(pd.Series(trade_returns)),
            "tp": int(((y_pred == 1) & (y_true == 1)).sum()),
            "fp": int(((y_pred == 1) & (y_true == 0)).sum()),
            "tn": int(((y_pred == 0) & (y_true == 0)).sum()),
            "fn": int(((y_pred == 0) & (y_true == 1)).sum()),
        })
    return rows


thresholds = np.arange(0.30, 0.71, 0.05)
threshold_rows = []
threshold_group_cols = ["instrument", "config_name", "feature_config", "feature_method", "model_name", "model_family"]

for keys, group in oof_predictions.groupby(threshold_group_cols, dropna=False):
    key_dict = dict(zip(threshold_group_cols, keys))
    for row in threshold_metric_rows(group, thresholds):
        threshold_rows.append({**key_dict, **row})

threshold_summary = pd.DataFrame(threshold_rows)
selected_thresholds = (
    threshold_summary
    .sort_values(["instrument", "config_name", "f1", "sharpe", "trade_count"], ascending=[True, True, False, False, False])
    .groupby(["instrument", "config_name", "feature_config", "model_name"], as_index=False, group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

display(threshold_summary.head(30))
display(selected_thresholds.head(20))


,instrument,config_name,feature_config,feature_method,model_name,model_family,threshold,accuracy,precision,recall,f1,trade_count,sharpe,tp,fp,tn,fn
0,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0.30,0.417957,0.379719,0.834146,0.521872,1351,10.340675,513,838,162,102
1,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0.35,0.463777,0.393734,0.756098,0.517817,1181,10.607027,465,716,284,150
2,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0.40,0.484830,0.394146,0.656911,0.492683,1025,9.926706,404,621,379,211
3,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0.45,0.528173,0.414035,0.575610,0.481633,855,9.776647,354,501,499,261
4,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0.50,0.562229,0.434844,0.499187,0.464799,706,9.592893,307,399,601,308
5,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0.55,0.585139,0.448790,0.391870,0.418403,537,8.256835,241,296,704,374
6,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0.60,0.600000,0.460358,0.292683,0.357853,391,6.848002,180,211,789,435
7,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0.65,0.607430,0.465201,0.206504,0.286036,273,5.100315,127,146,854,488
8,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0.70,0.622910,0.519737,0.128455,0.205997,152,4.693807,79,73,927,536
9,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t180_depth6_leaf10_mf50,random_forest,0.30,0.437771,0.381759,0.769106,0.510248,1239,10.611380,473,766,234,142


,instrument,config_name,feature_config,feature_method,model_name,model_family,threshold,accuracy,precision,recall,f1,trade_count,sharpe,tp,fp,tn,fn
0,cl1s,ewma_10d_tp2_sl2,pca90,pca,rf_t120_depth4_leaf5_sqrt,random_forest,0.3,0.412384,0.380029,0.860163,0.527155,1392,10.243691,529,863,137,86
1,cl1s,ewma_10d_tp2_sl2,pca90,pca,rf_t180_depth6_leaf10_mf50,random_forest,0.3,0.434056,0.385265,0.816260,0.523462,1303,10.284839,502,801,199,113
2,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t120_depth4_leaf5_sqrt,random_forest,0.3,0.417957,0.379719,0.834146,0.521872,1351,10.340675,513,838,162,102
3,cl1s,ewma_10d_tp2_sl2,cluster_corr90,cluster,rf_t180_depth6_leaf10_mf50,random_forest,0.3,0.437771,0.381759,0.769106,0.510248,1239,10.611380,473,766,234,142


## 8. Fit final selected models and inspect importance

Tree `feature_importances_` are used as MDI-style importances. For correlation clustering, importances are reported at the selected cluster representative level; for PCA, component importance is paired with top original-feature loadings.

In [10]:
def feature_importance_frame(model, feature_names: list[str]) -> pd.DataFrame:
    if hasattr(model, "feature_importances_"):
        importances = np.asarray(model.feature_importances_, dtype=float)
    else:
        raise AttributeError("The fitted model does not expose feature_importances_.")

    out = pd.DataFrame({"feature": feature_names, "importance": importances})
    total = out["importance"].sum()
    out["importance_share"] = out["importance"] / total if total > 0 else np.nan
    return out.sort_values("importance", ascending=False).reset_index(drop=True)


def cluster_level_importance(feature_importance: pd.DataFrame, prep_metadata: dict) -> pd.DataFrame:
    if prep_metadata["method"] != "cluster":
        return pd.DataFrame()

    selector = prep_metadata["selector"]
    cluster_summary = selector.cluster_summary_.copy()
    representative_to_members = {}
    representative_to_cluster_id = {}

    if not cluster_summary.empty:
        for row in cluster_summary.itertuples(index=False):
            representative_to_members[row.representative_feature] = list(row.cluster_features)
            representative_to_cluster_id[row.representative_feature] = int(row.cluster_id)

    rows = []
    for row in feature_importance.itertuples(index=False):
        members = representative_to_members.get(row.feature, [row.feature])
        is_forced_singleton = row.feature in ALWAYS_KEEP_FEATURES and row.feature not in representative_to_members
        rows.append({
            "cluster_id": representative_to_cluster_id.get(row.feature, np.nan),
            "representative_feature": row.feature,
            "cluster_size": len(members),
            "cluster_features": members,
            "cluster_note": "forced_singleton_core_signal" if is_forced_singleton else "correlation_cluster" if row.feature in representative_to_members else "singleton",
            "mdi_importance": row.importance,
            "mdi_importance_share": row.importance_share,
        })
    return pd.DataFrame(rows).sort_values("mdi_importance", ascending=False).reset_index(drop=True)


final_models = {}
feature_importance_tables = {}
cluster_importance_tables = {}
pca_loading_tables = {}

for _, selected in best_by_instrument.iterrows():
    label_row = label_files[
        (label_files["instrument"] == selected["instrument"])
        & (label_files["config_name"] == selected["config_name"])
    ].iloc[0]
    spec = next(s for s in model_specs if s["model_name"] == selected["model_name"])
    feature_config = next(cfg for cfg in FEATURE_SELECTION_CONFIGS if cfg["name"] == selected["feature_config"])

    data, feature_cols, embargo_observations = make_model_data(label_row)
    y = data["metalabel"].astype(int)

    X, _, transformed_feature_names, prep_metadata = prepare_fold_features(
        train=data,
        test=data,
        feature_cols=feature_cols,
        y_train=y,
        feature_config=feature_config,
    )

    model = make_estimator(spec, y)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", RuntimeWarning)
        model.fit(X, y)

    importance = feature_importance_frame(model, transformed_feature_names)
    feature_importance_tables[selected["instrument"]] = importance

    final_models[selected["instrument"]] = {
        "model": model,
        "preprocessor": prep_metadata,
        "raw_feature_cols": feature_cols,
        "transformed_feature_names": transformed_feature_names,
        "selected_config": selected.to_dict(),
    }

    print(f"{selected['instrument']} | {selected['config_name']} | {selected['feature_config']} | {selected['model_name']}")
    print("Rows:", len(data), "Raw features:", len(feature_cols), "Model features:", len(transformed_feature_names), "Embargo:", embargo_observations)
    print("Final fit warnings:", len(caught))
    print("Top model importances")
    display(importance.head(25))

    if prep_metadata["method"] == "cluster":
        cluster_importance = cluster_level_importance(importance, prep_metadata)
        cluster_importance_tables[selected["instrument"]] = cluster_importance
        print("Top MDI cluster importances")
        display(cluster_importance.head(25))
        print("Correlation clusters selected on final fit")
        display(prep_metadata["selector"].cluster_summary_.head(25))

    if prep_metadata["method"] == "pca":
        reducer = prep_metadata["reducer"]
        pca_loading_tables[selected["instrument"]] = reducer.get_top_loadings(n=8)
        print("PCA explained variance summary")
        display(reducer.pca_summary_.head(20))
        print("Top PCA loadings")
        display(pca_loading_tables[selected["instrument"]].head(40))


cl1s | ewma_10d_tp2_sl2 | cluster_corr90 | rf_t180_depth6_leaf10_mf50
Rows: 323 Raw features: 144 Model features: 50 Embargo: 10
Final fit warnings: 0
Top model importances


,feature,importance,importance_share
0,vol_ratio_63_126d,0.108851,0.108851
1,bb_upper_dist,0.107691,0.107691
2,sma50_slope,0.104854,0.104854
3,ret_spread_20_63,0.089802,0.089802
4,sma50_vs_sma100,0.069032,0.069032
5,sma100_vs_sma200,0.060422,0.060422
6,vol_126d,0.046138,0.046138
7,atr_5d,0.038325,0.038325
8,vol_252d,0.035083,0.035083
9,uo,0.032756,0.032756


Top MDI cluster importances


,cluster_id,representative_feature,cluster_size,cluster_features,cluster_note,mdi_importance,mdi_importance_share
0,NaN,vol_ratio_63_126d,1,[vol_ratio_63_126d],singleton,0.108851,0.108851
1,NaN,bb_upper_dist,1,[bb_upper_dist],singleton,0.107691,0.107691
2,5.0,sma50_slope,33,"[ret_20d, ret_63d, price_vs_sma5, price_vs_sma...",correlation_cluster,0.104854,0.104854
3,NaN,ret_spread_20_63,1,[ret_spread_20_63],singleton,0.089802,0.089802
4,NaN,sma50_vs_sma100,1,[sma50_vs_sma100],singleton,0.069032,0.069032
5,6.0,sma100_vs_sma200,4,"[price_vs_sma200, sma50_vs_sma200, sma100_vs_s...",correlation_cluster,0.060422,0.060422
6,NaN,vol_126d,1,[vol_126d],singleton,0.046138,0.046138
7,10.0,atr_5d,4,"[atr_5d, atr_10d, atr_14d, atr_20d]",correlation_cluster,0.038325,0.038325
8,NaN,vol_252d,1,[vol_252d],singleton,0.035083,0.035083
9,NaN,uo,1,[uo],singleton,0.032756,0.032756


Correlation clusters selected on final fit


,cluster_id,n_features,representative_feature,dropped_features,cluster_features,max_abs_corr_in_cluster,mean_abs_corr_in_cluster,selection_method
0,1,4,high,"[open, low, close]","[open, high, low, close]",0.996584,0.993023,target_corr
1,2,3,open_interest_zscore_63d,"[open_interest, open_interest_zscore_20d]","[open_interest, open_interest_zscore_20d, open...",0.906808,0.880954,target_corr
2,3,6,body_pct,"[ret_1d, co_range, rsi_7d_change, rsi_14d_chan...","[ret_1d, co_range, rsi_7d_change, rsi_14d_chan...",0.997927,0.949826,target_corr
3,4,2,ret_5d,[ret_5d_x_hmm_confidence],"[ret_5d, ret_5d_x_hmm_confidence]",0.996517,0.996517,target_corr
4,5,33,sma50_slope,"[ret_20d, ret_63d, price_vs_sma5, price_vs_sma...","[ret_20d, ret_63d, price_vs_sma5, price_vs_sma...",1.000000,0.716204,target_corr
5,6,4,sma100_vs_sma200,"[price_vs_sma200, sma50_vs_sma200, ema20_vs_em...","[price_vs_sma200, sma50_vs_sma200, sma100_vs_s...",0.956697,0.891998,target_corr
6,7,18,atr_5d_pct,"[vol_5d, vol_10d, vol_20d, ewma_vol_5d, ewma_v...","[vol_5d, vol_10d, vol_20d, ewma_vol_5d, ewma_v...",0.999680,0.823568,target_corr
7,8,4,vol_63d,"[ewma_vol_63d, park_vol_63d, gk_vol_63d]","[vol_63d, ewma_vol_63d, park_vol_63d, gk_vol_63d]",0.989921,0.934450,target_corr
8,9,2,true_range,[hl_range],"[hl_range, true_range]",0.994583,0.994583,target_corr
9,10,4,atr_5d,"[atr_10d, atr_14d, atr_20d]","[atr_5d, atr_10d, atr_14d, atr_20d]",0.986931,0.936156,target_corr


## 9. Optional save

In [11]:
if SAVE_OUTPUTS:
    import joblib

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path_level_results.to_csv(OUTPUT_DIR / "path_level_results.csv", index=False)
    baseline_path_results.to_csv(OUTPUT_DIR / "baseline_path_results.csv", index=False)
    baseline_summary.to_csv(OUTPUT_DIR / "baseline_summary.csv", index=False)
    oof_predictions.to_csv(OUTPUT_DIR / "oof_predictions.csv", index=False)
    threshold_summary.to_csv(OUTPUT_DIR / "threshold_summary.csv", index=False)
    selected_thresholds.to_csv(OUTPUT_DIR / "selected_thresholds.csv", index=False)
    candidate_summary.to_csv(OUTPUT_DIR / "candidate_summary.csv", index=False)
    comparison_to_baseline.to_csv(OUTPUT_DIR / "comparison_to_baseline.csv", index=False)
    best_by_instrument.to_csv(OUTPUT_DIR / "selected_configs.csv", index=False)
    best_by_instrument_config.to_csv(OUTPUT_DIR / "best_by_instrument_config.csv", index=False)

    for instrument, model_bundle in final_models.items():
        joblib.dump(model_bundle, OUTPUT_DIR / f"{instrument}_final_model.joblib")
        feature_importance_tables[instrument].to_csv(OUTPUT_DIR / f"{instrument}_feature_importance.csv", index=False)
        if instrument in cluster_importance_tables:
            cluster_importance_tables[instrument].to_csv(OUTPUT_DIR / f"{instrument}_cluster_importance.csv", index=False)
        if instrument in pca_loading_tables:
            pca_loading_tables[instrument].to_csv(OUTPUT_DIR / f"{instrument}_pca_top_loadings.csv", index=False)

    print("Saved outputs to", OUTPUT_DIR)
else:
    print("SAVE_OUTPUTS is False, so nothing was written.")


SAVE_OUTPUTS is False, so nothing was written.
